In [15]:
from pathlib import Path
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from dotenv import load_dotenv
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
load_dotenv()

True

In [3]:
pdf_path = Path("attention_u_need.pdf")

In [4]:
# Load this file in python program
loader = PyPDFLoader(file_path=pdf_path)

In [5]:
docs = loader.load()
print(docs[12])

page_content='Attention Visualizations
Input-Input Layer5
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
It
is
in
this
spirit
that
a
majority
of
American
governments
have
passed
new
laws
since
2009
making
the
registration
or
voting
process
more
difficult
.
<EOS>
<pad>
<pad>
<pad>
<pad>
<pad>
<pad>
Figure 3: An example of the attention mechanism following long-distance dependencies in the
encoder self-attention in layer 5 of 6. Many of the attention heads attend to a distant dependency of
the verb ‘making’, completing the phrase ‘making...more difficult’. Attentions here shown only for
the word ‘making’. Different colors represent different heads. Best viewed in color.
13' metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T

In [6]:
# Split the docs into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=400
)

chunks = text_splitter.split_documents(documents=docs)

In [7]:
print(chunks[0])

page_content='Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser ∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the Transformer,
based solely on attention mechanisms, dispensing with recurrence and convolution

In [13]:
# Vector Embeddings
embedding_model = HuggingFaceEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 984.54it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    host="localhost",
    port=8000,
    collection_name="learn_rag"
)

print("Indexing Done")

Indexing Done


In [17]:
client = chromadb.HttpClient(host="localhost", port=8000)
collection = client.get_collection("learn_rag")

In [18]:
print("Total docs:", collection.count())

Total docs: 63


In [21]:
print(collection.peek(3))

{'ids': ['4f390272-8307-4605-a313-7ab02e42221b', '8c12fd27-1163-4214-b7bf-ace18f76a94b', '4a65a7e5-22d7-41c6-a1b3-508cb8a2303c'], 'embeddings': array([[-0.11415933, -0.12801808,  0.03414962, ...,  0.08499372,
        -0.01825307, -0.05117758],
       [-0.08108726, -0.12717059,  0.03792082, ...,  0.03709139,
        -0.08084907, -0.04466775],
       [-0.05238722, -0.09842766,  0.01585094, ...,  0.03298934,
        -0.08638316,  0.01237836]], shape=(3, 384)), 'metadatas': [{'producer': 'pdfTeX-1.40.25', 'total_pages': 15, 'source': 'attention_u_need.pdf', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'creator': 'LaTeX with hyperref', 'author': '', 'moddate': '2024-04-10T21:11:43+00:00', 'page_label': '1', 'creationdate': '2024-04-10T21:11:43+00:00', 'title': '', 'page': 0, 'trapped': '/False', 'subject': '', 'keywords': ''}, {'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 